# MAGIC classification using saved SING results

COmmand to run:

docker run --rm --init \
  -v "$PWD:/workspace" -w /workspace \
  mhe_sing sh -c '
    pip install --no-cache-dir nbconvert ipykernel &&
    python -m nbconvert --to notebook --execute --inplace \
      --ExecutePreprocessor.kernel_name=python3 \
      --ExecutePreprocessor.timeout=-1 \
      sing_experiments/sing_magic_classification.ipynb
  '

Run the expensive analysis on the server with `python -u sing_experiments/run_magic_classification.py`. It fits both classes for **order 2 first**, then **order 3** in a fresh process, and saves all predictions, log densities, graph scores, coefficients, split indices, scaling, and solver diagnostics. Terminal output includes optimizer and derivative-batch progress. Completed classes and orders are reused when rerunning with the same configuration.

For nonparanormal runs, use `python -u sing_experiments/run_magic_classification_nonparanormal.py`. It also runs order 2 then order 3, writing to the separate `results/magic_classification_nonparanormal/` directory.

This notebook only loads saved results; it does not fit SING or require TransportMaps or Torch. Run all cells to inspect orders 2 and 3 for both preprocessing variants. Missing or failed runs are reported explicitly.

The models use the exact GTM fitting/validation/test row split (first ceil(2/3) of each class for the fitting/validation pool, five Torch-randomized folds with validation fold 4 and seed 25). Scaling uses pooled **fitting-fold** means and standard deviations. The original runs use no rank transform. The nonparanormal runs first apply a shared empirical Gaussianization fitted on the pooled fitting-fold rows only, followed by pooled training mean/std standardization. Average-rank midpoint probabilities handle ties; held-out values use interpolated training CDF knots with clipped tails. Neither validation nor test data determines the transformation. Class priors are equal. Original-run log densities are on the original observation scale. Nonparanormal log densities are on the **Gaussianized coordinate scale**, including the subsequent standardization Jacobian; the empirical transform does not supply an original-data density Jacobian. Do not compare raw likelihood values across these two scales. The shared preprocessing is identical for both class models, so their log-density difference remains the classification score. Validation/test observations never enter fitting.

In [ ]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'sing_experiments/run_magic_classification.py').exists())
RESULTS_DIR = ROOT / 'sing_experiments/results/magic_classification'
NONPARANORMAL_RESULTS_DIR = ROOT / 'sing_experiments/results/magic_classification_nonparanormal'
DELTA = 1.0
FIT_SEED = 25
DATA_PATH = ROOT / 'demos/data/magic04.csv'
DATA_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() if DATA_PATH.exists() else None

## Load and display a completed order

ROC-AUC and the ROC curve rank held-out observations by `log p_g(x) − log p_h(x)` (gamma is the positive class). Binary metrics use posterior probability 0.5, equivalent to log odds ≥ 0. At each fixed FPR, the TPR table reports the largest TPR with FPR less than or equal to that level, without interpolation.

The saved summary includes full configuration, package versions, SING commit, split metadata, and per-iteration solver diagnostics. `arrays.npz` includes exact split indices, all class/split log densities, graph/score matrices, map coefficients, and predictions; fitted Python density objects are not needed for the analysis below.

In [ ]:
def analyze_order(order, nonparanormal=False):
    results_dir = NONPARANORMAL_RESULTS_DIR if nonparanormal else RESULTS_DIR
    variant = 'nonparanormal' if nonparanormal else 'standardization only'
    label = f'SING order {order} ({variant})'
    runner = 'run_magic_classification_nonparanormal.py' if nonparanormal else 'run_magic_classification.py' 
    prefix = f'p{order}_delta{DELTA:g}_seed{FIT_SEED}_'
    if DATA_SHA256 is not None:
        candidates = [results_dir / f'{prefix}{DATA_SHA256[:12]}']
    else:
        candidates = sorted(results_dir.glob(f'{prefix}*'))
    if len(candidates) > 1:
        raise ValueError(f'Multiple datasets for order {order}; select a specific RESULTS_DIR or restore the source CSV.')
    if not candidates or not (candidates[0] / 'summary.json').exists():
        print(f'{label}: no completed results. Run {runner} --orders {order}.')
        if candidates and (candidates[0] / 'status.json').exists():
            display(json.loads((candidates[0] / 'status.json').read_text()))
        return None
    directory = candidates[0]
    status_path = directory / 'status.json'
    if status_path.exists() and json.loads(status_path.read_text())['status'] != 'ok':
        print(f'{label}: run is incomplete.')
        display(json.loads(status_path.read_text()))
        return None
    summary = json.loads((directory / 'summary.json').read_text())
    assert summary['config']['order'] == order
    assert bool(summary['config'].get('nonparanormal_transform', False)) == nonparanormal
    if DATA_SHA256 is not None:
        assert summary['split_metadata']['sha256'] == DATA_SHA256
    with np.load(directory / 'arrays.npz', allow_pickle=False) as archive:
        arrays = {key: archive[key].copy() for key in archive.files}
    tables = {name: pd.read_csv(directory / f'{name}.csv') for name in
              ['test_predictions', 'roc_curve', 'tpr_at_fpr', 'density_loglik']}
    tables['fit_summary'] = pd.read_csv(directory / 'fit_summary.csv', index_col=0)
    tables['feature_scaling'] = pd.read_csv(directory / 'feature_scaling.csv', index_col=0)
    scores = pd.DataFrame([summary['classification_metrics']], index=[label])
    split_counts = pd.DataFrame({g: {s: len(arrays[f'{g}_{s}_row_indices'])
                                     for s in ['train', 'validate', 'test']} for g in ['g', 'h']}).T
    print(f'Loaded {label}: {directory}')
    print('Configuration:')
    display(pd.Series(summary['config'], name='value').to_frame())
    print('Split counts and training-only scaling:')
    display(split_counts, tables['feature_scaling'])
    print(f'Fit summary and density diagnostics ({summary["loglik_scale"]}):')
    display(tables['fit_summary'], tables['density_loglik'])
    print('Classification scores and TPR at fixed FPR:')
    display(scores, tables['tpr_at_fpr'].rename(columns={'SING': label}))
    roc = tables['roc_curve']
    predictions = tables['test_predictions']
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
    axes[0].plot(roc.fpr, roc.tpr,
                 label=f'{label}, AUC={summary["classification_metrics"]["roc_auc"]:.3f}')
    axes[0].plot([0, 1], [0, 1], '--', color='#94a3b8', linewidth=1)
    axes[0].set(xlabel='False-positive rate (hadron classified as gamma)',
                ylabel='True-positive rate (gamma detected)', title=f'MAGIC test ROC — {label}',
                xlim=(0, 1), ylim=(0, 1.01))
    axes[0].legend(loc='lower right')
    axes[0].grid(alpha=0.2)
    ConfusionMatrixDisplay(confusion_matrix(predictions.gamma_label, predictions.predicted_gamma_label,
                                            labels=[0, 1]),
                           display_labels=['Hadron (h)', 'Gamma (g)']).plot(
        ax=axes[1], colorbar=False, cmap='Blues')
    axes[1].set_title(f'{variant}: posterior threshold 0.5')
    plt.show()
    return dict(label=label, nonparanormal=nonparanormal, directory=directory, summary=summary, arrays=arrays, scores=scores, **tables)

## Order 2: complete analysis

In [ ]:
order2 = analyze_order(2)

## Order 3: complete analysis

In [ ]:
order3 = analyze_order(3)

## Nonparanormal preprocessing: order 2

Shared training-fitted Gaussianization, then standardization. Likelihoods use Gaussianized coordinates.


In [ ]:
order2_nonparanormal = analyze_order(2, nonparanormal=True)


## Nonparanormal preprocessing: order 3


In [ ]:
order3_nonparanormal = analyze_order(3, nonparanormal=True)


## Compare preprocessing variants and orders

Classification metrics, TPR at fixed FPR, and ROC curves compare all available runs on the same held-out rows. Exact split indices are checked across variants; scaling and fitted nonparanormal CDF knots are checked between orders within each variant. Raw-data and Gaussianized-coordinate likelihoods are reported separately above and are **not directly comparable**. No order or preprocessing variant is selected using test performance.

The loaded `order2`, `order3`, `order2_nonparanormal`, and `order3_nonparanormal` dictionaries retain predictions, ROC points, graph scores, coefficients, densities, and solver diagnostics. Nonparanormal transformation knots are saved separately in each run's `preprocessing.npz` for reuse.


In [ ]:
available = [result for result in [order2, order3, order2_nonparanormal, order3_nonparanormal]
             if result is not None]
if available:
    reference = available[0]
    for result in available[1:]:
        assert reference['summary']['split_metadata']['sha256'] == result['summary']['split_metadata']['sha256']
        for group in ['g', 'h']:
            for split in ['train', 'validate', 'test']:
                key = f'{group}_{split}_row_indices'
                np.testing.assert_array_equal(reference['arrays'][key], result['arrays'][key])
    for nonparanormal in [False, True]:
        same_preprocessing = [r for r in available if r['nonparanormal'] == nonparanormal]
        if len(same_preprocessing) == 2:
            left, right = same_preprocessing
            for key in ['training_mean', 'training_std']:
                np.testing.assert_array_equal(left['arrays'][key], right['arrays'][key])
            if nonparanormal:
                with np.load(left['directory'] / 'preprocessing.npz', allow_pickle=False) as a, \
                     np.load(right['directory'] / 'preprocessing.npz', allow_pickle=False) as b:
                    assert set(a.files) == set(b.files)
                    for key in a.files:
                        np.testing.assert_array_equal(a[key], b[key])
    display(pd.concat([result['scores'] for result in available]))
    comparison = available[0]['tpr_at_fpr'][['FPR']].copy()
    for result in available:
        comparison = comparison.merge(result['tpr_at_fpr'].rename(columns={'SING': result['label']}),
                                      on='FPR', validate='one_to_one')
    display(comparison)
    fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
    for result in available:
        roc = result['roc_curve']
        ax.plot(roc.fpr, roc.tpr, label=result['label'])
    ax.plot([0, 1], [0, 1], '--', color='#94a3b8', linewidth=1)
    ax.set(xlabel='False-positive rate', ylabel='True-positive rate', title='MAGIC test ROC comparison',
           xlim=(0, 1), ylim=(0, 1.01))
    ax.legend(loc='lower right')
    ax.grid(alpha=0.2)
    plt.show()
else:
    print('No completed MAGIC analyses yet.')
